# TsSims — 时间序列数据模拟工具包

本 Notebook 演示 `TsSims` 包的完整用法，包括：

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Ts.TsSims import (
    simulate_sarima, simulate_garch,
    simulate_gjr_garch, simulate_egarch, simulate_garch_m,
    simulate_trend_stationary, simulate_difference_stationary,
    simulate_cointegrated,
)

---
## 1. SARIMA 过程模拟

### 1.1 AR(1) — 一阶自回归

In [ ]:
r = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.7], seed=42)
print(r.summary())
r.plot()
plt.show()

### 1.2 MA(1) — 一阶移动平均

In [ ]:
r = simulate_sarima(n=200, order=(0, 0, 1), ma=[0.5], seed=123)
print(r.summary())
r.plot()
plt.show()

### 1.3 ARMA(2,1) — 混合模型

In [ ]:
r = simulate_sarima(n=200, order=(2, 0, 1), ar=[0.5, -0.3], ma=[0.4], seed=99)
print(r.summary())
r.plot()
plt.show()

### 1.4 ARIMA(1,1,1) — 带差分的非平稳过程

In [ ]:
r = simulate_sarima(
    n=200, order=(1, 1, 1),
    ar=[0.3], ma=[0.5], const=0.1,   # drift = 0.1
    seed=42,
)
print(r.summary())
r.plot()
plt.show()

### 1.5 季节性 SARIMA(1,0,1)(1,0,1,4) — 季度数据

In [ ]:
r = simulate_sarima(
    n=200,
    order=(1, 0, 1), ar=[0.5], ma=[0.3],
    seasonal_order=(1, 0, 1, 4), seasonal_ar=[0.4], seasonal_ma=[0.2],
    seed=7,
)
print(r.summary())
r.plot()
plt.show()

### 1.6 SimSARIMAResult 方法

In [ ]:
r = simulate_sarima(n=100, order=(1, 0, 0), ar=[0.7], seed=42)

# 获取 Series
s = r.get_data()
print(type(s), s.head(3))

# 获取参数 (深拷贝)
p = r.get_params()
print(p)

# 验证深拷贝
p["ar"] = [0.99]
print(r.get_params()["ar"])  # 仍为 [0.7]

## 2. 纯 ARCH 过程 (GARCH with q=0)

### 2.1 ARCH(1) — GARCH(p=1, q=0)

In [ ]:
r = simulate_garch(n=300, p=1, q=0, omega=0.4, alpha=[0.5], seed=42)
print(r.summary())
r.plot()
plt.show()

# 检查峰度 — ARCH 产生厚尾
print(f"Kurtosis: {pd.Series(r.data).kurtosis():.3f} (Normal = 0)")

### 2.2 ARCH(2) — GARCH(p=2, q=0)

In [ ]:
r = simulate_garch(n=500, p=2, q=0, omega=0.2, alpha=[0.3, 0.2], seed=10)
r.plot()
plt.show()

### 2.3 ARCH(1) with Student's t 新息 — GARCH(p=1, q=0)

In [ ]:
r_norm = simulate_garch(n=500, p=1, q=0, dist="normal", seed=42)
r_t    = simulate_garch(n=500, p=1, q=0, dist="t", dist_params={"df": 5}, seed=42)

### 2.4 ARCH with AR mean — GARCH(p=1, q=0, mean_model="ar")

In [ ]:
r = simulate_garch(
    n=300, p=1, q=0, omega=0.4, alpha=[0.5],
    mean_model="ar", mean_ar=[0.6], mean_const=1.0,
    seed=42,
)

---
## 3. GARCH 过程模拟

### 3.1 GARCH(1,1) — 经典波动率模型

In [ ]:
r = simulate_garch(
    n=300, p=1, q=1,
    omega=0.1, alpha=[0.2], beta=[0.7],
    seed=42,
)
print(r.summary())
r.plot()
plt.show()

# 持久性
print(f"α + β = {0.2 + 0.7:.2f} → 高持久性")

### 3.2 GARCH(1,2) — 更多 GARCH 滞后

In [ ]:
r = simulate_garch(
    n=300, p=1, q=2,
    omega=0.1, alpha=[0.2], beta=[0.4, 0.3],
    seed=42,
)
r.plot()
plt.show()

### 3.3 SimGARCHResult 方法 — to_dataframe()

In [ ]:
r = simulate_garch(n=200, p=1, q=1, seed=42)
df = r.to_dataframe()
print(df.head(10))
# 验证: data = mean_const + errors
print(f"\ndata ≈ errors check: {np.allclose(df['data'], df['residuals'], atol=1e-10)}")


### 3.4 非平稳 GARCH 会报错

In [ ]:
try:
    simulate_garch(n=100, p=1, q=1, omega=0.2, alpha=[0.6], beta=[0.5], seed=42)
except ValueError as e:
    print(f"ValueError: {e}")

---
## 3.5 GJR-GARCH — 非对称 GARCH（杠杆效应）

GJR-GARCH(1,1,1): 负面冲击对波动率影响更大（杠杆效应）。

```python
# sigma2_t = omega + alpha*eps2_{t-1} + gamma*I_{t-1}*eps2_{t-1} + beta*sigma2_{t-1}
# 其中 I_{t-1} = 1 if eps_{t-1} < 0 else 0
```

In [ ]:
from Ts.TsSims import simulate_gjr_garch

r = simulate_gjr_garch(
    n=300, p=1, q=1, o=1,
    omega=0.05, alpha=[0.10], gamma=[0.15], beta=[0.70],
    seed=42,
)
print(r.summary())
r.plot()
plt.show()

# 验证: alpha + 0.5*gamma + beta 决定平稳性
p = r.get_params()
persistence = sum(p["alpha"]) + 0.5 * sum(p["gamma"]) + sum(p["beta"])
print(f"Persistence (alpha + 0.5*gamma + beta) = {persistence:.3f}")

---
## 3.6 EGARCH — 指数 GARCH（对数方差建模）

EGARCH(1,1,1): 通过对数方差天然保证方差为正，支持杠杆效应。



In [ ]:
from Ts.TsSims import simulate_egarch

r = simulate_egarch(
    n=300, p=1, q=1, o=1,
    omega=0.0, alpha=[0.20], gamma=[0.10], beta=[0.30],
    seed=42,
)
print(r.summary())
r.plot()
plt.show()

---
## 3.7 GARCH-M — ARCH-in-Mean（波动率进入均值方程）

GARCH-M(1,1): 条件波动率 sigma_t 进入均值方程，
反映风险-收益权衡关系。



In [ ]:
from Ts.TsSims import simulate_garch_m

r = simulate_garch_m(
    n=300, p=1, q=1,
    omega=0.10, alpha=[0.20], beta=[0.60],
    garch_m_kappa=0.20, garch_m_form="vol",
    seed=42,
)
print(r.summary())
r.plot()
plt.show()

# 对比: kappa=0 等价于标准 GARCH
r0 = simulate_garch_m(n=300, p=1, q=1, omega=0.10,
                       alpha=[0.20], beta=[0.60],
                       garch_m_kappa=0.0, seed=42)
r_g = simulate_garch(n=300, p=1, q=1, omega=0.10,
                     alpha=[0.20], beta=[0.60], seed=42)
print(f"kappa=0 yields same data as standard GARCH: {np.allclose(r0.data, r_g.data)}")

---
## 4. TS / DS 过程

两种典型的非平稳时间序列。

### 4.1 趋势平稳 (TS) — 冲击影响是暂时的

In [ ]:
r_ts = simulate_trend_stationary(
    n=200, intercept=10.0, slope=0.5, sigma=2.0, seed=42,
)
print(r_ts.summary())
r_ts.plot()
plt.show()

# 去除趋势后应平稳
from statsmodels.tsa.tsatools import detrend
detrended = detrend(r_ts.data, order=1)
print(f"去趋势后 std = {detrended.std():.3f} (接近 sigma=2.0)")

### 4.2 差分平稳 (DS) — 冲击影响是永久的

In [ ]:
r_ds = simulate_difference_stationary(
    n=200, drift=1.0, sigma=2.0, seed=42, burn=100,
)
print(r_ds.summary())
r_ds.plot()
plt.show()

# 一阶差分后应平稳
diff_data = np.diff(r_ds.data)
print(f"一阶差分后 mean = {diff_data.mean():.3f} (接近 drift=1.0)")
print(f"一阶差分后 std  = {diff_data.std():.3f} (接近 sigma=2.0)")

### 4.3 TS vs DS 对比图

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.plot(r_ts.data, linewidth=1, label="TS")
ax1.set_title("Trend-Stationary (TS)\ny_t = t + ε_t")
ax1.legend()
ax2.plot(r_ds.data, linewidth=1, color="#D55E00", label="DS")
ax2.set_title("Difference-Stationary (DS)\ny_t = 1 + y_{t-1} + ε_t")
ax2.legend()
plt.tight_layout()
plt.show()

---
## 6. 种子可复现性

### 5.2 k=3, r=2 — 三个变量，两个协整关系

设计: y2 为共同随机趋势，y0-y2 和 y1-y2 为平稳的协整关系。

In [ ]:
from statsmodels.tsa.stattools import adfuller

# ============================================================
# 三变量系统 (k=3, r=2)
#   y2 为共同随机趋势（不修正），y0-y2 和 y1-y2 为平稳的协整关系
# ============================================================
alpha = np.array([
    [-0.4,  0.0],   # y0: 对 spread1 修正
    [ 0.0, -0.3],   # y1: 对 spread2 修正
    [ 0.0,  0.0],   # y2: 弱外生（共同趋势）
])
beta = np.array([
    [ 1.0,  0.0],   # 协整向量1: y0 - y2 ~ I(0)
    [ 0.0,  1.0],   # 协整向量2: y1 - y2 ~ I(0)
    [-1.0, -1.0],
])

r3 = simulate_cointegrated(n=500, k=3, coint_rank=2,
                            alpha=alpha, beta=beta, sigma=0.5, seed=42)
print(r3.summary())
r3.plot()
plt.show()

# 验证两个 spread 的平稳性
df3 = r3.get_data()
spread1 = df3["y0"] - df3["y2"]
spread2 = df3["y1"] - df3["y2"]

_, p1, *_ = adfuller(spread1)
_, p2, *_ = adfuller(spread2)
print(f"Spread1 (y0 - y2) ADF p-value: {p1:.4f} → {'平稳 ✓' if p1 < 0.05 else '非平稳 ✗'}")
print(f"Spread2 (y1 - y2) ADF p-value: {p2:.4f} → {'平稳 ✓' if p2 < 0.05 else '非平稳 ✗'}")

In [ ]:
# ============================================================
# 可视化: 共同趋势 vs 平稳 spread
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(10, 5))

# 原始 I(1) 序列
axes[0, 0].plot(df3["y0"], linewidth=1, label="y0")
axes[0, 0].plot(df3["y1"], linewidth=1, label="y1")
axes[0, 0].plot(df3["y2"], linewidth=1, label="y2 (common trend)")
axes[0, 0].set_title("Raw Series (I(1))")
axes[0, 0].legend(fontsize=8)

# Spread 1: y0 - y2 (I(0))
axes[0, 1].plot(spread1, linewidth=1, color="#009E73")
axes[0, 1].axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
axes[0, 1].set_title("Spread: y0 - y2 (I(0))")

# Spread 2: y1 - y2 (I(0))
axes[1, 0].plot(spread2, linewidth=1, color="#CC79A7")
axes[1, 0].axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
axes[1, 0].set_title("Spread: y1 - y2 (I(0))")

# 协整空间散点图
axes[1, 1].scatter(spread1, spread2, s=5, alpha=0.6)
axes[1, 1].set_xlabel("y0 - y2")
axes[1, 1].set_ylabel("y1 - y2")
axes[1, 1].set_title("Cointegration Space")
plt.tight_layout()
plt.show()

### 5.3 稳定性验证 — 不稳定 VECM 会报错

`simulate_cointegrated` 内部检查 VECM 稳定性条件：
$$|\lambda_i(I_r + \beta'\alpha)| < 1$$

若特征值在单位圆外或圆上，抛出 `ValueError`。

In [ ]:
# ============================================================
# 不稳定 VECM: alpha 太大 → |eig(I + beta'@alpha)| >= 1
# ============================================================
try:
    alpha_unstable = np.array([[-1.5], [0.0]])
    beta_unstable  = np.array([[1.0], [-1.0]])
    simulate_cointegrated(n=100, k=2, coint_rank=1,
                          alpha=alpha_unstable, beta=beta_unstable, seed=42)
except ValueError as e:
    print(f"ValueError: {e}")

# 验证特征值
M = np.eye(1) + beta_unstable.T @ alpha_unstable
eigval = np.linalg.eigvals(M)[0]
print(f"|eig(I + beta'@alpha)| = {abs(eigval):.4f} (应 >= 1)")

## 小结

| 函数 | 用途 | 结果类型 |
|------|------|----------|
| `simulate_sarima()` | SARIMA / ARMA 过程 | `SimSARIMAResult` |
| `simulate_garch()` | GARCH(p,q) / ARCH(p) 波动率 | `SimGARCHResult` |
| `simulate_gjr_garch()` | GJR-GARCH(p,o,q) 杠杆效应 | `SimGARCHResult` |
| `simulate_egarch()` | EGARCH(p,o,q) 对数方差 | `SimGARCHResult` |
| `simulate_garch_m()` | GARCH-M (ARCH-in-Mean) | `SimGARCHResult` |
| `simulate_igarch()` | IGARCH(p,q) 单位根波动率 | `SimGARCHResult` |
| `simulate_cointegrated()` | 协整多变量系统 (VECM) | `SimCointegratedResult` |
| `simulate_trend_stationary()` | 趋势平稳 (TS) 序列 | `SimTSDSResult` |
| `simulate_difference_stationary()` | 差分平稳 (DS) 序列 | `SimTSDSResult` |

**关键设计**:
- `GARCH(p, q=0)` = 纯 ARCH(p)，通过 `q=0` 切换
- GJR-GARCH 引入杠杆效应 (gamma 系数)
- EGARCH 对数方差天然保证方差为正
- GARCH-M 波动率进入均值方程
- 协整模拟通过 VECM 表示 `Delta Y = alpha @ beta.T @ Y + epsilon`，与 TsModels.VECM 估计直接对应
- 支持 Student's t 厚尾新息
- 所有结果类继承 `BaseSimResult`，提供 `get_data()`, `get_params()`, `summary()`, `plot()`

---
## 5. 协整多变量过程模拟

通过 VECM 表示生成 k 维协整时间序列：
$$\Delta Y_t = \alpha \beta' Y_{t-1} + \varepsilon_t$$

### 5.1 k=2, r=1 — 两个变量，一个协整关系

In [ ]:
# ============================================================
# 示例 A: 默认 alpha / beta
#   alpha = -0.5 * I_r（上半部分），beta = [I_r; 0]
#   即 y0 对自身偏离均衡做出修正，y1 不受影响（弱外生）
# ============================================================
r = simulate_cointegrated(n=300, k=2, coint_rank=1, seed=42)
print(r.summary())
r.plot()
plt.show()

# 验证: spread = y0 - y1 应为平稳过程
from statsmodels.tsa.stattools import adfuller

df = r.get_data()
spread = df["y0"] - df["y1"]
adf_stat, adf_pval, *_ = adfuller(spread)
print(f"Spread (y0 - y1) ADF 检验: stat = {adf_stat:.3f}, p-value = {adf_pval:.4f}")
print(f"→ {'拒绝单位根 — 协整关系成立 ✓' if adf_pval < 0.05 else '无法拒绝单位根 ✗'}")

In [ ]:
# ============================================================
# 示例 B: 自定义 alpha / beta — y1 弱外生
#   alpha = [[-0.3], [0.0]] → 只有 y0 对误差修正做出反应
#   beta  = [[1.0], [-1.0]] → 协整关系: y0 - y1 ~ I(0)
# ============================================================
alpha = np.array([[-0.3], [0.0]])
beta  = np.array([[1.0], [-1.0]])
r2 = simulate_cointegrated(n=500, k=2, coint_rank=1,
                            alpha=alpha, beta=beta, seed=123)
r2.plot()
plt.show()
print(r2.summary())

# 验证弱外生: y1 不受误差修正影响 (alpha[1] = 0)
p = r2.get_params()
print(f"alpha[0,0] = {p['alpha'][0,0]:.3f} (y0 修正速度)")
print(f"alpha[1,0] = {p['alpha'][1,0]:.3f} (y1 修正速度 ← 弱外生)")

In [ ]:
r1 = simulate_sarima(n=50, order=(1, 0, 0), seed=42)
r2 = simulate_sarima(n=50, order=(1, 0, 0), seed=42)
r3 = simulate_sarima(n=50, order=(1, 0, 0), seed=99)

print(f"Same seed identical: {np.allclose(r1.data, r2.data)}")
print(f"Diff seed different: {not np.allclose(r1.data, r3.data)}")

---
## 7. Rational distributed lag (RDL) 模拟与估计恢复

`RDLInputSpec` 的键给出活动多项式滞后；模拟结果的 `distributed_lags` 可直接传给估计器。

In [ ]:
from Ts.TsModels import SARIMAX
from Ts.TsSims import RDLInputSpec, simulate_rdl

rdl = simulate_rdl(
    n=600,
    distributed_lags={
        "x1": RDLInputSpec(numerator={0: 1.2}, denominator={1: 0.45}),
        "x2": RDLInputSpec(numerator={0: -0.7}, denominator={1: 0.20}),
    },
    sigma2=0.09,
    seed=2305,
)
print(rdl.summary())
display(rdl.get_exog().head())
display(rdl.get_components().head())

estimated = SARIMAX(
    rdl.data,
    exog=rdl.get_exog(),
    order=(0, 0, 0),
    trend="n",
    distributed_lags=rdl.distributed_lags,
).fit(method="bfgs", maxiter=300, require_convergence=True)
display(estimated.distributed_lag_coefficients)
display(estimated.steady_state_gains)

assert abs(estimated.params["rdl.x1.omega.L0"] - 1.2) < 0.07
assert abs(estimated.params["rdl.x1.delta.L1"] - 0.45) < 0.05
assert abs(estimated.params["rdl.x2.omega.L0"] + 0.7) < 0.07
assert abs(estimated.params["rdl.x2.delta.L1"] - 0.20) < 0.07
print("TsSims -> TsModels RDL recovery checks passed.")